## Notebook 07 – Evaluation Modell A (AP 3.7): Konzeptioneller Neuansatz

### Anlass

Vor Beginn der eigentlichen Evaluationsarbeit wurde das zugrunde liegende
Geschaeftsprozessverstaendnis kritisch hinterfragt und praezisiert - mit
wichtigen Konsequenzen fuer die Interpretation und den Nutzen von
Modell A.

### Praezisiertes Prozessverstaendnis (Extruder GmbH als Maschinenhersteller)

1. **Auftragseingang:** Kunde (Rohrhersteller) bestellt eine individuelle
   Extrusionsanlage mit Vorgaben (Material, Geometrie-/Durchmesserbereich,
   Stueckzahl)
2. **Konzeptionelle Freigabe:** Experte waehlt grob Extrudertyp/
   Groessenklasse basierend auf Erfahrung (eigener Entscheidungsschritt,
   bisher NICHT Teil des Projektumfangs)
3. **Bau:** Maschine wird konzeptionell/mechanisch konfiguriert und gebaut
   (individuell pro Auftrag, keine Serienmaschine)
4. **Inbetriebnahme/Erstlauf:** Experte waehlt erfahrungsbasierten
   Startpunkt fuer Prozessparameter, beobachtet Ergebnis, korrigiert
   iterativ bei Fehlern (z.B. "Rohr wird oval" -> "Vakuum erhoehen"),
   bis Qualitaet passt
5. **Auslieferung:** Maschine wird mit finalen Parametern an Kunden
   ausgeliefert, IO-Bericht dokumentiert Ergebnis

### Kritische Neubewertung: Wofuer ist Modell A tatsaechlich nuetzlich?

**Urspruengliche Annahme (Projektbeginn):** Modell A dient als Filter -
sortiert aus vielen historischen (Einstellung, Ergebnis)-Paaren die
guten heraus, damit Modell B nur aus erfolgreichen Faellen lernt.

**Problem mit dieser Annahme:** Da zu jeder ausgelieferten Maschine
bereits ein IO-Bericht existiert, ist die Information "war diese
Einstellung gut" bereits bekannt - ein Modell, das dies nachtraeglich
vorhersagt, loest kein reales Problem (die Information liegt schon vor).

**Neue, praezisere Rolle fuer Modell A:** Unterstuetzung des iterativen
Korrekturschritts WAEHREND der Inbetriebnahme (Schritt 4) - konkret:

- **Multi-Label-Variante von Modell A** (bereits in Notebook 05/06
  trainiert): sagt nicht nur IO/NIO vorher, sondern WELCHES der 9
  Einzelkriterien voraussichtlich verletzt wird (Wandstaerke, Ovalitaet,
  Bindenaehte, ...) - das ist die "Fehlerdiagnose"-Faehigkeit des
  scheidenden Experten, die bisher nicht explizit als Kernwert erkannt
  wurde
- **SHAP** (bereits als Pflicht-Feature fuer AP 3.7/Streamlit-Demonstrator
  vorgesehen, bisher nur als "Erklaerbarkeit" verstanden): liefert pro
  Vorhersage, WELCHER Parameter am staerksten zur NIO-Einschaetzung
  beitraegt - das ist naeherungsweise die "welche Korrektur wuerde
  helfen"-Faehigkeit des Experten

**Modell B** bleibt fuer den ERSTEN Schritt zustaendig (Auftrag -> guter
Startpunkt), noch zu bauen. Der iterative Korrekturschritt danach wird
durch Modell A (Diagnose) + SHAP (Korrekturrichtung) unterstuetzt - eine
Kombination, die vorher nicht als zusammenhaengendes Konzept erkannt war.

### Offene, kritische Frage vor Nutzung dieses Konzepts

Die Multi-Label-Gesamtguete (Macro-F1 ~0.42, Notebook 05/06) verdeckt
moeglicherweise grosse Unterschiede zwischen den 9 Einzelkriterien -
insbesondere die seltenen (Bindenaehte <1%, Blasenbildung ~2%) koennten
grundsaetzlich nicht robust diagnostizierbar sein (Shannon-Entropie-
Grenze, Notebook 03), waehrend haeufigere Kriterien (Ovalitaet 33.9%
Anteil an NIO-Faellen) moeglicherweise gut funktionieren.

**Ziel dieses Notebooks (Kernprinzip der Studie, nicht Ergebnis-
Maximierung):** ehrlich, datengetrieben zeigen, FUER WELCHE Fehlertypen
eine automatisierte Diagnose+Korrektur-Unterstuetzung mit der aktuellen
Datenlage machbar ist - und fuer welche nicht. Eine Machbarkeitsstudie
liefert Wert durch eine ehrliche "geht/geht nicht"-Aussage, nicht durch
moeglichst hohe Kennzahlen.

In [1]:
# =============================================================================
# Zelle 02 – Setup & Daten laden
# =============================================================================
import sys
sys.path.append('../src')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE
from preprocessing import load_dataset, get_target, get_feature_set, FEATURE_SETS, TARGET_VARIANTS, DNResidualizer, NUMERISCHE_BASIS_SPALTEN, KATEGORIALE_SPALTEN

SEED = 42
apply_store44_style()

df = load_dataset("../data/processed/model_a_preprocessed.csv")

# --- Identische Reproduktion Train/Test-Split und aeussere Folds (Notebook 05/06) ---
from sklearn.model_selection import train_test_split, StratifiedKFold

train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df["io_nio"], random_state=SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(train_idx, df.loc[train_idx, "io_nio"]))
fold_splits_idx = [(train_idx[tr_pos], train_idx[val_pos]) for tr_pos, val_pos in fold_splits]

print(f"Datensatz: {df.shape}")
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}, Folds: {len(fold_splits_idx)}")

Datensatz: (700, 44)
Train: 560, Test: 140, Folds: 5


In [3]:
# =============================================================================
# Zelle 03 – Pipeline-Builder + ALLE DREI Shortlist-Kandidaten (Sane-Default)
# =============================================================================
# KORREKTUR: nicht nur GaussianNB, sondern alle drei Multi-Label-Shortlist-
# Kandidaten aus Notebook 05 (GaussianNB, LogisticRegression, SVC) - keine
# vorschnelle Einzelentscheidung. Sane-Default-Hyperparameter aus Notebook 05
# verwendet (NICHT die in Notebook 06 verworfenen, instabilen getunten
# Werte fuer multilabel). Feature-Set je Modell aus Notebook 05 uebernommen
# (datengetrieben, nicht neu geraten).
# =============================================================================

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

def baue_preprocessing_pipeline(feature_set_name):
    numerisch_transform = Pipeline([
        ("select_numerisch", FunctionTransformer(lambda X: X[NUMERISCHE_BASIS_SPALTEN])),
        ("scale", StandardScaler()),
    ])
    kategorial_transform = Pipeline([
        ("select_kategorial", FunctionTransformer(lambda X: X[KATEGORIALE_SPALTEN].astype(float))),
        ("scale", StandardScaler()),
    ])
    residual_transform = Pipeline([
        ("residualize", DNResidualizer(numerische_spalten=NUMERISCHE_BASIS_SPALTEN)),
        ("scale", StandardScaler()),
    ])
    if feature_set_name == "original":
        return FeatureUnion([("numerisch", numerisch_transform), ("kategorial", kategorial_transform)])
    elif feature_set_name == "original_no_kategorial":
        return numerisch_transform
    elif feature_set_name == "residual":
        return FeatureUnion([("residual", residual_transform), ("kategorial", kategorial_transform)])
    elif feature_set_name == "combined":
        return FeatureUnion([("numerisch", numerisch_transform), ("residual", residual_transform), ("kategorial", kategorial_transform)])
    raise ValueError(f"Unbekanntes Feature-Set: {feature_set_name}")

# --- Alle drei Shortlist-Kandidaten, Sane-Default-Konfiguration (Notebook 05) ---
MULTILABEL_KANDIDATEN = {
    "GaussianNB": {"modell": GaussianNB(), "feature_set": "original_no_kategorial"},
    "LogisticRegression": {"modell": LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear", random_state=SEED), "feature_set": "original"},
    "SVC": {"modell": SVC(class_weight="balanced", random_state=SEED), "feature_set": "original"},
}

for name, konfig in MULTILABEL_KANDIDATEN.items():
    print(f"{name}: Feature-Set={konfig['feature_set']}, Modell={konfig['modell']}")

GaussianNB: Feature-Set=original_no_kategorial, Modell=GaussianNB()
LogisticRegression: Feature-Set=original, Modell=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                   solver='liblinear')
SVC: Feature-Set=original, Modell=SVC(class_weight='balanced', random_state=42)


In [5]:
# =============================================================================
# Zelle 04 – Einzelkriterien-Diagnosegueete, alle 3 Shortlist-Kandidaten
# =============================================================================
# Kernfrage der Machbarkeitsstudie: fuer WELCHE der 9 NIO-Kriterien ist
# eine automatisierte Diagnose datenseitig tragfaehig - unabhaengig vom
# Modell, oder modellabhaengig unterschiedlich?
# =============================================================================

LABEL_NAMEN = ["y_nio_wandstaerke", "y_nio_ovalitaet", "y_nio_od", "y_nio_wellhoehe",
               "y_nio_bindenaehte", "y_nio_blasenbildung", "y_nio_risse",
               "y_nio_oberflaechenfehler", "y_nio_delamination"]

y_full, valid_mask = get_target(df, "multilabel", respect_mask=True)

alle_ergebnisse = []
for modell_name, konfig in MULTILABEL_KANDIDATEN.items():
    fs = konfig["feature_set"]
    for label_idx, label_name in enumerate(LABEL_NAMEN):
        fold_f1 = []
        for tr_idx, val_idx in fold_splits_idx:
            prep = baue_preprocessing_pipeline(fs)
            X_tr = prep.fit_transform(df.loc[tr_idx])
            X_val = prep.transform(df.loc[val_idx])
            y_tr = y_full.loc[tr_idx].values[:, label_idx]
            y_val = y_full.loc[val_idx].values[:, label_idx]

            modell = clone(konfig["modell"])
            modell.fit(X_tr, y_tr)
            y_pred = modell.predict(X_val)
            fold_f1.append(f1_score(y_val, y_pred, zero_division=0))

        positiv_rate = y_full[label_name].mean()
        alle_ergebnisse.append({
            "modell": modell_name, "kriterium": label_name,
            "f1_mean": round(np.mean(fold_f1), 3), "f1_std": round(np.std(fold_f1), 3),
            "positiv_rate_prozent": round(positiv_rate*100, 2),
        })

einzel_df = pd.DataFrame(alle_ergebnisse)
pivot = einzel_df.pivot(index="kriterium", columns="modell", values="f1_mean")
pivot["positiv_rate_prozent"] = einzel_df.groupby("kriterium")["positiv_rate_prozent"].first()
pivot = pivot.sort_values("positiv_rate_prozent", ascending=False)

print(pivot.to_string())
einzel_df.to_csv("../reports/tables/07_einzelkriterien_diagnose_guete.csv", index=False)
print("\nGespeichert: reports/tables/07_einzelkriterien_diagnose_guete.csv")

modell                    GaussianNB  LogisticRegression    SVC  positiv_rate_prozent
kriterium                                                                            
y_nio_ovalitaet                0.205               0.220  0.203                  7.71
y_nio_oberflaechenfehler       0.049               0.157  0.105                  6.14
y_nio_wellhoehe                0.115               0.151  0.121                  4.14
y_nio_wandstaerke              0.145               0.122  0.061                  3.71
y_nio_od                       0.052               0.116  0.063                  3.71
y_nio_blasenbildung            0.071               0.067  0.071                  2.00
y_nio_risse                    0.067               0.076  0.044                  1.14
y_nio_delamination             0.000               0.040  0.100                  0.86
y_nio_bindenaehte              0.000               0.051  0.200                  0.71

Gespeichert: reports/tables/07_einzelkriterien_diagno

In [8]:
# =============================================================================
# Zelle 05 – Theoretischer Bayes-Floor PRO EINZELKRITERIUM (Monte-Carlo)
# =============================================================================
# Analog zu Notebook 03, Zelle 14 (dort: kombinierte io_nio-Zielgroesse),
# jetzt fuer jedes der 9 Einzelkriterien separat. Beantwortet: ist die
# schwache erreichte F1 (Zelle 04) "schlecht trotz viel Potenzial" oder
# "nahe am tatsaechlich Machbaren"?
# KORREKTUR: df (Preprocessing-Datensatz) enthaelt kalibriermechanismus/
# wandtyp bereits One-Hot-encodiert (Spalten mechanismus_Vakuum,
# wandtyp_einwandig) - fuer die Simulation werden die ROHEN kategorialen
# Werte benoetigt, daher zusaetzlich data/raw/model_a_raw.csv geladen.
# =============================================================================

df_raw_fuer_simulation = pd.read_csv("../data/raw/model_a_raw.csv")
df_latent_ref = pd.read_csv("../data/raw/model_a_latent_reference.csv")

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

rng_sim = np.random.default_rng(999)
M = 2000
N_full = len(df)

duesenspalt_ = df["duesenspalt"].values
massetemperatur_ = df["massetemperatur"].values
mfr_charge_ = df["mfr_charge"].values
abzugsgeschwindigkeit_ = df["abzugsgeschwindigkeit"].values
kuehlwassertemperatur_ = df["kuehlwassertemperatur"].values
kalibriermechanismus_ = df_raw_fuer_simulation["kalibriermechanismus"].values
kalibrierdruck_mbar_ = df_raw_fuer_simulation["kalibrierdruck_mbar"].values
wandtyp_ = df_raw_fuer_simulation["wandtyp"].values
rohr_dn_ = df_latent_ref["rohr_dn_latent"].values
wandstaerke_ideal_ = df_latent_ref["wandstaerke_ideal_latent"].values

druck_norm_ = np.where(
    kalibriermechanismus_ == "Vakuum",
    np.clip((np.abs(kalibrierdruck_mbar_) - 100) / 800, 0, 1),
    np.clip((kalibrierdruck_mbar_ - 20) / 380, 0, 1),
)

p_kriterium = {name: np.zeros(N_full) for name in LABEL_NAMEN}

for m in range(M):
    die_swell_real = np.clip(0.85 + 0.003*(massetemperatur_-205) - 0.05*(mfr_charge_-0.7) + rng_sim.normal(0,0.015,N_full), 0.70, 1.00)
    wandstaerke_ist_sim = duesenspalt_ / die_swell_real
    sigma_od = 0.8 - 0.5*druck_norm_
    aussendurchmesser_ist_sim = rohr_dn_ + rng_sim.normal(0, sigma_od, N_full)
    ovalitaet_sim = np.abs(rng_sim.normal(0, sigma_od*0.6, N_full))
    ausformungsgrad_sim = np.clip(0.85 + 0.15*druck_norm_ + rng_sim.normal(0,0.03,N_full), 0.6, 1.05)

    score_bindenaht = -3.0 + np.clip((200-massetemperatur_)/6, -4, 4)
    bindenaht = rng_sim.random(N_full) < sigmoid(score_bindenaht)
    score_blase = -3.2 + np.clip((massetemperatur_-222)/5, -4, 4)
    blase = rng_sim.random(N_full) < sigmoid(score_blase)
    score_riss = -3.2 + np.clip((11-kuehlwassertemperatur_)/4,-4,4) + 0.4*np.clip((abzugsgeschwindigkeit_-9)/3,-4,4)
    riss = rng_sim.random(N_full) < sigmoid(score_riss)
    score_oberflaeche = -3.0 + np.clip((np.abs(wandstaerke_ist_sim-wandstaerke_ideal_)-0.15)/0.08,-4,4)
    oberflaeche = rng_sim.random(N_full) < sigmoid(score_oberflaeche)
    score_delam = -3.48 + np.clip((215-massetemperatur_)/8,-4,4)
    delam = rng_sim.random(N_full) < sigmoid(score_delam)
    delam = np.where(wandtyp_=="doppelwandig", delam, False)

    nio_wandstaerke = np.abs(wandstaerke_ist_sim-wandstaerke_ideal_) > 0.30
    nio_ovalitaet = ovalitaet_sim > 0.65
    nio_od = np.abs(aussendurchmesser_ist_sim-rohr_dn_) > 1.2
    nio_wellhoehe = ausformungsgrad_sim < 0.85

    p_kriterium["y_nio_wandstaerke"] += nio_wandstaerke
    p_kriterium["y_nio_ovalitaet"] += nio_ovalitaet
    p_kriterium["y_nio_od"] += nio_od
    p_kriterium["y_nio_wellhoehe"] += nio_wellhoehe
    p_kriterium["y_nio_bindenaehte"] += bindenaht
    p_kriterium["y_nio_blasenbildung"] += blase
    p_kriterium["y_nio_risse"] += riss
    p_kriterium["y_nio_oberflaechenfehler"] += oberflaeche
    p_kriterium["y_nio_delamination"] += delam

for k in p_kriterium:
    p_kriterium[k] /= M

# --- Bayes-optimales F1 je Kriterium (bester Schwellenwert) ---
bayes_ergebnisse = []
for label_name, p in p_kriterium.items():
    if label_name == "y_nio_delamination":
        gueltig = df_raw_fuer_simulation["wandtyp"] == "doppelwandig"
        p = p[gueltig.values]
    best_f1, best_thr = 0, 0.5
    for thr in np.arange(0.02, 0.98, 0.02):
        pred = p >= thr
        TP = np.sum(p[pred]); FP = np.sum(1-p[pred]); FN = np.sum(p[~pred])
        prec = TP/(TP+FP) if (TP+FP)>0 else 0
        rec = TP/(TP+FN) if (TP+FN)>0 else 0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    bayes_ergebnisse.append({"kriterium": label_name, "bayes_f1_optimal": round(best_f1,3), "bayes_schwelle": round(best_thr,2)})

bayes_df = pd.DataFrame(bayes_ergebnisse).set_index("kriterium")
print(bayes_df)
bayes_df.to_csv("../reports/tables/07_bayes_floor_je_kriterium.csv")

                          bayes_f1_optimal  bayes_schwelle
kriterium                                                 
y_nio_wandstaerke                    0.811            0.38
y_nio_ovalitaet                      0.172            0.08
y_nio_od                             0.126            0.06
y_nio_wellhoehe                      0.208            0.10
y_nio_bindenaehte                    0.031            0.02
y_nio_blasenbildung                  0.096            0.04
y_nio_risse                          0.095            0.06
y_nio_oberflaechenfehler             0.334            0.16
y_nio_delamination                   0.115            0.06
